# Locomotive Analyzer — Google Colab Training

این Notebook از همان اسکریپت `training/train_model.py` مخزن استفاده می‌کند؛ بنابراین منطق آموزش و ارزیابی با نسخه لوکال یکسان است.

مراحل: دریافت مخزن → نصب کتابخانه‌ها → آپلود Excel → آموزش و ارزیابی → دانلود ONNX bundle و گزارش.

In [ ]:
import os, pathlib, subprocess, sys
REPO_URL = "https://github.com/SharedByQwen/LocomotiveAnalayzer.git"
WORKDIR = pathlib.Path("/content/LocomotiveAnalayzer")
if WORKDIR.exists():
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
os.chdir(WORKDIR)
print("Repository:", WORKDIR)


In [ ]:
%pip install -r requirements-train.txt


In [ ]:
from google.colab import files
uploaded = files.upload()
excel_files = [name for name in uploaded if name.lower().endswith((".xlsx", ".xls"))]
if not excel_files:
    raise ValueError("Please upload the locomotive Excel dataset (.xlsx).")
dataset_path = WORKDIR / "training" / "data" / "LocomotivDataSet.xlsx"
dataset_path.parent.mkdir(parents=True, exist_ok=True)
dataset_path.write_bytes(pathlib.Path(excel_files[0]).read_bytes())
print("Dataset saved to:", dataset_path)


In [ ]:
result = subprocess.run(
    [sys.executable, "training/train_model.py", "--dataset", str(dataset_path)],
    cwd=WORKDIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {result.returncode}")
print("Training completed successfully.")


In [ ]:
from google.colab import files
outputs = [
    WORKDIR / "backend" / "models" / "model_bundle.zip",
    WORKDIR / "training" / "output" / "evaluation_report.json",
    WORKDIR / "training" / "README.md",
]
for output in outputs:
    if output.exists():
        print("Downloading", output.name)
        files.download(str(output))
    else:
        print("Missing output:", output)
